# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** This notebook turns the ML-02 research question
into a task type, a target, and a metric — with the code that backs each claim.

Continues from `w01_research_question.ipynb`.

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is computed from the 30-day impression pair.
# So trend_direction, trend_pct, impressions_last_30d and impressions_prev_30d are never features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")


Working dir: /content/Rayanflyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421


## 1. My lane as an ML task (type)

**Ranking / scoring**, not classification.

The decision my output feeds is "which pages does an editor open first, given room for 20–50 this
sprint?" That is an ordering problem under a capacity constraint. The output is a priority score per
content item; the artifact a human touches is a ranked queue with reason codes.

I do fit a binary classifier underneath (declining vs not), but I use its probability **as a ranking
score** and I evaluate it as a ranking (precision@K), not as a classifier at a 0.5 threshold. The
distinction matters for how I report results: nobody at FlyRank cares about accuracy across 30,000
pages, because nobody is going to act on 30,000 pages. Only the top of the list gets acted on, so
only the top of the list should be scored.

It is **not** clustering — I am not asking "what kinds of pages exist", I am asking "which ones
first", and I have a label available to order against. It is **not** plain signal analysis — the
output has to be an actionable ordered list, not a set of effect sizes.

The one-paragraph frame:

> For a FlyRank **content strategist**, deciding **which published pages get this sprint's limited
> refresh hours**, I will build a **ranked review queue with reason codes** from the **anonymized
> starter dataset (30,000 pseudonymized pages, trailing 90 days)**, scoring **each page's likelihood
> of being in measured organic decline**, measured by **precision@K against the 54.2% base rate**. A
> wrong call costs **an editorial slot spent on a stable page (~$300–$1,000) or a decaying revenue
> page left unreviewed**. A plain rule isn't enough because **the signals are individually weak and
> partly non-monotone** (Section 5 measures this). I will claim only **observed / measured /
> decision-support** results — never that a refresh causes recovery.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Why "ranking" is the right task type: the action is capacity-limited, so only the top matters.
CAPACITY_PER_SPRINT = 50

print(f"Pages in the portfolio:            {len(df):,}")
print(f"Pages an editor can review/sprint: {CAPACITY_PER_SPRINT}")
print(f"Share of the portfolio acted on:   {CAPACITY_PER_SPRINT / len(df) * 100:.2f}%")
print()
print("=> The metric must score the top of the list, not the whole list.")

Pages in the portfolio:            30,000
Pages an editor can review/sprint: 50
Share of the portfolio acted on:   0.17%

=> The metric must score the top of the list, not the whole list.


## 2. Target or proxy

**Target:** `is_declining_label` = 1 when `trend_direction == "down"`.

**Where it comes from — and the honest problem with it.** This label is **defined by a rule, not
observed as an outcome**. The dataset computes `trend_direction` from the last-30-day vs
previous-30-day impression change: `down` means the drop was worse than −20%. So my label is a
threshold on a ratio of two columns that sit in the same file I am modelling from.

Two consequences I have to carry through the whole project rather than bury:

1. **It is a proxy for "in decline", not for "worth refreshing".** Nobody at FlyRank observed an
   editor's judgement or a recovery outcome here. A page can be flagged `down` and still not be worth
   an editorial slot (off-season intent, a deliberate deprecation), and a page can be `stable` and
   badly need a rewrite. The model learns the dataset's definition of decline. That is a real ceiling
   on what any result here can mean.
2. **It creates a hard leakage boundary.** Because the label is a deterministic function of
   `impressions_last_30d` and `impressions_prev_30d`, those two columns reconstruct the label
   **exactly** — the cell below shows 1.0000 agreement. So the excluded list is not just
   `trend_direction` / `trend_pct` (the two the docs name); it must also exclude the 30-day impression
   pair, and anything derived from them.

**The label I would prefer, and why I am not using it yet.** The honest version is a past→future
label: features from a window ending at time *T*, outcome measured in a *later* window (e.g.
impressions in the 30 days after *T*). The starter CSV is one trailing-90-day snapshot, so it cannot
express that — the "future" window is inside the same file. The warehouse release
(`fact_content_daily_performance`, 2025-01-27 → 2026-06-30) can, and that is the upgrade path named in
my limitations. Everything I report from the starter CSV is therefore **concurrent** decline detection
("this page looks like the declining ones"), not forecasting ("this page will decline").

The measured check below also tests the *clicks* and *sessions* 30-day pairs. They agree with the label
only at roughly the base rate, so they are correlated context rather than label sources. I exclude them
anyway: same ratio shape as the label, weak signal, not worth defending in a review.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Can the 30-day column pairs reconstruct the label? (down := pct change < -20%, prev == 0 -> not down)
def reconstruct_label(last: pd.Series, prev: pd.Series) -> np.ndarray:
    pct = np.where(prev > 0, (last - prev) / prev.where(prev > 0) * 100.0, np.nan)
    return np.where(prev == 0, 0, np.where(pct < -20.0, 1, 0))


print(f"Label base rate (share 'down'): {BASE_RATE:.4f}")
print()
print("Agreement of each 30-day pair with the label:")
for metric in ("impressions", "clicks", "sessions"):
    recon = reconstruct_label(df[f"{metric}_last_30d"], df[f"{metric}_prev_30d"])
    label = f"{metric}_last_30d / {metric}_prev_30d"
    print(f"  {label:45s} {(recon == y).mean():.4f}")

impressions_recon = reconstruct_label(df["impressions_last_30d"], df["impressions_prev_30d"])
assert (impressions_recon == y).mean() == 1.0, "expected the impression pair to reconstruct the label"

print()
print("=> impressions pair = 1.0000: it IS the label. Excluded as leakage.")
print("=> clicks / sessions pairs land near the base rate: correlated context, not the label.")
print("   Excluded anyway - same ratio shape as the label, not worth defending.")

Label base rate (share 'down'): 0.5421

Agreement of each 30-day pair with the label:
  impressions_last_30d / impressions_prev_30d   1.0000
  clicks_last_30d / clicks_prev_30d             0.5364
  sessions_last_30d / sessions_prev_30d         0.5383

=> impressions pair = 1.0000: it IS the label. Excluded as leakage.
=> clicks / sessions pairs land near the base rate: correlated context, not the label.
   Excluded anyway - same ratio shape as the label, not worth defending.


## 3. Success metric

**Primary metric: precision@50, reported next to the 54.2% base rate.** Secondary: precision@20 and
precision@100 (the plausible range of sprint capacity), plus ROC-AUC as the overall discrimination
number.

**Why precision@K.** K is the editor's capacity. Precision@50 answers exactly the operational
question: *of the 50 pages this system puts at the top, how many were genuinely in measured decline?*
Recall is close to meaningless here — with 16,262 declining pages and 50 slots, the maximum achievable
recall is about 0.3%, so optimising recall would be theatre.

**What "good" means — stated before training, not after.**

| Threshold | precision@50 | Reading |
|---|---|---|
| Floor (must clear) | **> 0.542** | At or below the base rate, the system is no better than picking pages at random. |
| Useful | **≥ 0.70** | About 35 of 50 slots land on genuinely declining pages — a real efficiency gain over random triage. |
| Target | **≥ 0.74** | Matches the reference pipeline's random-forest number (`outputs/model_report.md`), so it is a bar I know is reachable on this data. |

**The base rate is the whole point of reporting it.** 54.2% of pages carry the positive label, so a
system that flags everything already scores 0.542. Any precision figure in this project appears next to
that number, or it is not a claim. Note what this implies about the repo's reference *rule* baseline at
precision@50 = 0.240: that is *less than half* the base rate, i.e. the rule actively mis-prioritises.
Beating 0.240 is not the achievement — beating 0.542 is.

**Why not accuracy or F1.** Both average over the 29,950 pages nobody will look at. A page ranked
17,000th and one ranked 29,000th are operationally identical (untouched), but accuracy treats moving one
of them across a threshold as progress.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# The metric, defined now (before any modelling), and its floor on this data.
# precision_at_k() is defined in the setup cell so every later notebook uses the same function.
MAX_RECALL_AT_50 = 50 / y.sum()

print(f"Base rate (flag-everything precision): {BASE_RATE:.4f}")
print(f"Positives in the data:                 {y.sum():,}")
print(f"Max achievable recall at K=50:         {MAX_RECALL_AT_50:.4%}  <- why recall is not the metric")
print()

# Sanity-check the metric against a random ranking: it must land near the base rate.
rng = np.random.default_rng(RANDOM_STATE)
trials = [precision_at_k(y, rng.random(len(df)), 50) for _ in range(200)]
print(f"Random ranking, precision@50 over 200 trials: mean {np.mean(trials):.4f} "
      f"(5th-95th pct {np.percentile(trials, 5):.2f}-{np.percentile(trials, 95):.2f})")
print(f"=> Metric behaves: random ~= base rate {BASE_RATE:.3f}. At or below this is not a win.")

SUCCESS_FLOOR = BASE_RATE
SUCCESS_USEFUL = 0.70
SUCCESS_TARGET = 0.74
print(f"\nCommitted thresholds -> floor {SUCCESS_FLOOR:.3f} | useful {SUCCESS_USEFUL:.2f} | "
      f"target {SUCCESS_TARGET:.2f}")

Base rate (flag-everything precision): 0.5421
Positives in the data:                 16,262
Max achievable recall at K=50:         0.3075%  <- why recall is not the metric

Random ranking, precision@50 over 200 trials: mean 0.5338 (5th-95th pct 0.44-0.62)
=> Metric behaves: random ~= base rate 0.542. At or below this is not a win.

Committed thresholds -> floor 0.542 | useful 0.70 | target 0.74


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item**, observed over one trailing 90-day window.

`content_id` is the unit; `client_id` is the portfolio it belongs to. Both are pseudonyms and both are
**context, never features** — `client_id`'s job in this project is to be the grouping key for the
train/test split.

The grain is verified below, not assumed: 30,000 rows, 30,000 distinct `content_id`s, zero duplicates.
So there is no page-day or page-query fan-out to collapse and no risk of double counting — this file is
already at the grain my decision needs. (The warehouse's `fact_content_daily_performance` is at
`report_date × client × content`, a different grain that *would* need aggregating before it could
answer this question — worth stating so the contrast is explicit.)

One thing the printout makes visible: the 30,000 pages are spread across 32 clients very unevenly, from
3 pages to 7,008 pages. That imbalance is why Section 5 and my split design care about client
grouping — a random row split would let a handful of large clients sit on both sides of the split.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Grain probe: one row per content_id, or the unit of analysis is wrong.
dupes = df.groupby("content_id").size().pipe(lambda s: s[s > 1])
print(f"Rows: {len(df):,} | distinct content_id: {df['content_id'].nunique():,} | duplicated: {len(dupes)}")
assert len(dupes) == 0, "grain broken: content_id is not unique"

per_client = df["client_id"].value_counts()
print(f"Clients: {df['client_id'].nunique()} | pages per client: "
      f"min {per_client.min()}, median {int(per_client.median())}, max {per_client.max():,}")
print()

# The unit of analysis, shown: decision-relevant columns for one row = one page.
UNIT_VIEW = [
    "content_id", "client_id", "content_type", "impressions_90d",
    "days_with_impressions", "avg_position", "ctr", "days_since_last_update",
    "content_age_days", "is_declining_label",
]
print("One row = one pseudonymized content item:")
print(df[UNIT_VIEW].head(3).to_string(index=False))

Rows: 30,000 | distinct content_id: 30,000 | duplicated: 0
Clients: 32 | pages per client: min 3, median 567, max 7,008

One row = one pseudonymized content item:
          content_id         client_id    content_type  impressions_90d  days_with_impressions  avg_position  ctr  days_since_last_update  content_age_days  is_declining_label
content_304f48230142 client_f369cb89fc keyword article             3803                     88          10.6 0.76                      20               187                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320                     88          20.3 0.05                      25               445                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581                     88          36.5 0.09                      20               141                   1


## 5. Why ML beats a fixed rule here

Honest answer up front: **a hand-tuned rule does well on this label, and I will not claim otherwise
until ML-08 measures both.** What I can show now is why a *naive* if-statement fails, and where a
learned model has room to win. Three measured reasons:

**1. The obvious rule has almost nothing to fire on.** The textbook refresh heuristic is "stale but
still getting traffic". In this slice only **174 pages** sit in the `181+` freshness tier, and only
**17** are both ≥180 days stale and ≥500 impressions. A rule gated on staleness cannot fill a 50-slot
queue — it fills 17 slots and then ranks the remaining 33 by whatever the tie-break happens to be. Any
precision@50 built that way is mostly an artifact of tie-breaking, not of the rule. (ML-07 shows this
happening for real.)

**2. The signals are individually weak and some are non-monotone.** `days_with_impressions` is the
single most informative field I found, and its relationship to the label *reverses*: pages with ≤4 days
of impressions decline at **14.9%**, pages with 46–69 days at **64.9%**, then it falls back to **56.2%**
at 88 days. A threshold rule must pick one direction; the truth is a band. Same shape on position:
`top_3` pages decline least (**24.1%**), `striking` (positions 11–20) most (**61.0%**), and `deep` pages
least again (**34.4%**). "Worse position ⇒ more decline" is simply false here.

**3. The intuitive signal is real but hidden — a single threshold cannot see it.** My ML-02 framing
assumed a CTR deficit would mark decaying pages. Measured **unconditionally**, that looks false: the
label rate across CTR bands moves only between **50.7% and 57.8%** against a 54.2% base rate. But
measured **among visible pages only** (`impressions_90d ≥ 1000`, n=13,512) the same signal is clean and
monotone: **0.677 → 0.601 → 0.552 → 0.470 → 0.460** as CTR rises.

The confound is that **13,212 rows have `ctr == 0`**, mostly tiny pages where CTR is a meaningless ratio
over a handful of impressions. They swamp the unconditional view. So CTR carries real signal *conditional
on volume* — which is precisely an interaction, not a threshold, and precisely what a fixed rule cannot
express while a model can. (Verdict recorded in `w04_signal_audit.ipynb`; my earlier "CTR is flat"
reading was measured on the wrong population.)

**Where that leaves ML.** The pattern is real but lives in *interactions and bands*
(established-but-imperfect coverage × mid-tier position × moderate volume × some staleness), not in
single thresholds. Finding those bands by hand means eyeballing crosstabs and hard-coding cut points —
which is fitting the data, just with worse bookkeeping and no held-out check. A model does the same job
with a grouped split and an honest metric attached. So the argument is **not "ML is stronger" but "ML is
the honest way to fit what I would otherwise hand-tune"** — and ML-07 builds the hand-tuned rule anyway,
precisely so ML-08 has a real bar to clear.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Evidence for Section 5: how little the naive rule can fire on, and where signals reverse.
print("(1) The naive 'stale but visible' rule has almost nothing to fire on")
print(df["freshness_tier"].value_counts(dropna=False).to_string())
naive_hits = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"  stale >=180d AND >=500 impressions: {naive_hits} rows -> cannot fill a 50-slot queue")
print()

print("(2) days_with_impressions: the relationship reverses (label rate by band)")
bands = pd.cut(df["days_with_impressions"], [0, 4, 20, 46, 69, 81, 87, 90])
print(df.groupby(bands, observed=True)["is_declining_label"].agg(["mean", "size"]).round(3).to_string())
print()
print("    position_tier: mid positions decline most, top_3 and deep least")
print(df.groupby("position_tier")["is_declining_label"].agg(["mean", "size"]).round(3).to_string())
print()

print(f"(3) CTR unconditionally looks flat against a base rate of {BASE_RATE:.3f} ...")
ctr_bands = pd.qcut(df["ctr"], 5, duplicates="drop")
print(df.groupby(ctr_bands, observed=True)["is_declining_label"].agg(["mean", "size"]).round(3).to_string())
print(f"    ... because {(df['ctr'] == 0).sum():,} rows have ctr == 0 (tiny pages, meaningless ratio)")
print()
visible = df[df["impressions_90d"] >= 1000].copy()
visible["ctr_band"] = pd.cut(visible["ctr"], [-0.01, 0.1, 0.25, 0.5, 1.0, 100])
print(f"    among VISIBLE pages only (impressions_90d >= 1000, n={len(visible):,}) the signal is monotone:")
print(visible.groupby("ctr_band", observed=True)["is_declining_label"].agg(["mean", "size"]).round(3).to_string())
print("    -> CTR carries signal CONDITIONAL on volume: an interaction, not a threshold.")
print()

print("Client effect (why the split must be grouped by client):")
by_client = df.groupby("client_id")["is_declining_label"].agg(["mean", "size"])
print(f"  label rate per client ranges {by_client['mean'].min():.3f} to {by_client['mean'].max():.3f} "
      f"across {len(by_client)} clients")
print("  => a random row split would leak client-level decline rates between train and test.")

(1) The naive 'stale but visible' rule has almost nothing to fire on
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
  stale >=180d AND >=500 impressions: 17 rows -> cannot fill a 50-slot queue

(2) days_with_impressions: the relationship reverses (label rate by band)
                        mean   size
days_with_impressions              
(0, 4]                 0.149   3157
(4, 20]                0.495   2924
(20, 46]               0.618   2992
(46, 69]               0.649   3088
(69, 81]               0.647   2983
(81, 87]               0.616   3864
(87, 90]               0.562  10992

    position_tier: mid positions decline most, top_3 and deep least
                mean   size
position_tier              
deep           0.344   1319
page_1         0.570  11814
page_3_5       0.562   7242
striking       0.610   7304
top_3          0.241   2321

(3) CTR unconditionally looks flat against a base rate of 0.542 ...
                 mean   size
ctr          

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.